# Guider ROI Quality (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-02-06  
**Last Modified:** 2026-03-24  
**Status:** In Progress  
**Keywords:** Guider, ROI, Location, Star Catalog, Pointing  

## Description

Check that the guider ROI was correctly located on each CCD, by comparing
stars found vs. the one expected from the guide star catalog.

Key functionality:
1. Load guider star catalog and ROI metadata for a range of exposures
2. Compare measured star positions with catalog predictions, accounting for pointing offsets
3. Identify correctly matched stars vs. volunteer (unexpected) stars
4. Produce diagnostic plots of ROI placement quality

**Output:** Diagnostic plots (PNG) and summary table (Parquet) in `output/`  
**Based on:** check_roi_quality-20260128.ipynb

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-02-06 | Aaron Roodman | Initial version (check_roi_quality-20260128.ipynb) |
| 2026-03-24 | Aaron Roodman | Revised to rubin-work template; added parameter cell, output directory, parquet export with date range |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Data Access](#data)
4. [Derived Quantities](#derived)
5. [Results & Plots](#results)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

day_obs = 20260128               # Observation date
seq_num_lo = 17                  # First sequence number (inclusive)
seq_num_hi = 235                 # Last sequence number (inclusive)

# Output directory
output_dir = 'output'

<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from astropy.table import Table, vstack, join

from lsst.afw import cameraGeom
from lsst.obs.lsst import LsstCam
from lsst.geom import Point2D, Extent2D
from lsst.summit.utils.guiders.transformation import pixelToFocal

# Add repo root to path for common imports
sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

# Add guider code to path
sys.path.insert(0, str(Path.cwd() / 'code'))
from guider_utils import *

setup_plotting()

camera = LsstCam.getCamera()
det_nums = {det.getName(): i for i, det in enumerate(camera)}

os.makedirs(output_dir, exist_ok=True)

In [ ]:
%load_ext autoreload
%autoreload 2

<a id='data'></a>
## Data Access

Load guider star catalog and ROI metadata for all science exposures in the selected range.

In [ ]:
allstar = getmany_guiderstarcat(day_obs, seq_num_lo, seq_num_hi)

In [ ]:
# Compute ROI star magnitude from flux
# RubinTV reports a zeropoint of 27.9 for z band images (per second).
# For 0.2 second integration, adjust by -2.5 * log10(1/0.2) = -1.75
# Convert to electrons from ADU using typical Gain of 1.45
allstar['magroi'] = -2.5 * np.log10(allstar['flux'] * 1.45) + 26.2

In [ ]:
allstar['detid'] = [det_nums[aname] for aname in allstar['ccdName']]

In [ ]:
parquet_name = f'guider_roi_quality_{day_obs}_{seq_num_lo}-{seq_num_hi}.parquet'
allstar.write(os.path.join(output_dir, parquet_name), format='parquet', overwrite=True)
print(f'Wrote {len(allstar)} rows to {output_dir}/{parquet_name}')

<a id='derived'></a>
## Derived Quantities

Compute pointing-corrected CCD offsets and classify stars as "found" (catalog match) or "volunteer" (unexpected).

In [ ]:
table = allstar
table['yroi_forccd'] = np.where(table['yccd'] < 2000, table['yroi'], -table['yroi'])

In [ ]:
# Convert delta_x, delta_y from pointing error to delta_xccd, delta_yccd
# Also account for delta_rot (arcsec)

llpt_ccd = Extent2D(0., 0.)  # just need rotation, no offset

detNames = table['ccdName'].tolist()
dxs = table['delta_x'].tolist()
dys = table['delta_y'].tolist()
xccds = table['xccd'].tolist()
yccds = table['yccd'].tolist()
delta_xccd = []
delta_yccd = []
drot = np.deg2rad(np.array(table['delta_rot'].tolist()) / 3600.0)
cosdrot = np.cos(drot)
sindrot = np.sin(drot)

for i, detName in tqdm(enumerate(detNames)):
    detector = camera[detName]

    # Additional offset from delta_rot
    if xccds[i] is not None:
        fpx, fpy = pixelToFocal(xccds[i], yccds[i], detector)
    else:
        fpx, fpy = pixelToFocal(2000., 2000., detector)
    dxrot = (fpx[0] * cosdrot[i] - fpy[0] * sindrot[i]) - fpx[0]
    dyrot = (fpx[0] * sindrot[i] + fpy[0] * cosdrot[i]) - fpy[0]

    # Convert from mm to pixel equivalent
    dxrot = dxrot / 0.01
    dyrot = dyrot / 0.01

    ft, bt = mk_ccd_to_dvcs(llpt_ccd, detector.getOrientation().getNQuarter())
    dxyccd = bt(Point2D(dxs[i] + dxrot, dys[i] + dyrot))

    delta_xccd.append(dxyccd.getX())
    delta_yccd.append(dxyccd.getY())

table['delta_xccd'] = delta_xccd
table['delta_yccd'] = delta_yccd

In [ ]:
# Classify stars
volunteer = (table['gaia_G_corr'] - table['magroi'] < -2.0)
found = (np.abs(table['gaia_G_corr'] - table['magroi']) < 2.0)

tabf = table[found]
tabvol = table[volunteer]

print(f'Found (catalog match): {found.sum()}, Volunteer: {volunteer.sum()}, Total: {len(table)}')

In [ ]:
# Setup per-detector colors for plots
cmap = plt.cm.tab10
det_colors = {}
i = 0
for detector in camera:
    if detector.getType() == cameraGeom.DetectorType.GUIDER:
        det_colors[detector.getName()] = cmap(i)
        i += 1

table['ccdColor'] = [det_colors[det] for det in table['ccdName']]

<a id='results'></a>
## Results & Plots

### Magnitude Comparison

In [ ]:
f, ax = plt.subplots(1, 1, figsize=(7, 7))
ax.scatter(allstar['gaia_G_corr'], allstar['magroi'], s=4.)
ax.set_aspect('equal')
ax.set_xlabel('Gaia Magnitude with Vignetting Correction')
ax.set_ylabel('ROI star magnitude')
ax.set_title(f'DayObs {day_obs} SeqNum {seq_num_lo}-{seq_num_hi}')
f.savefig(os.path.join(output_dir, f'guider_mag_roi_vs_catalog_{day_obs}_{seq_num_lo}-{seq_num_hi}.png'))

In [ ]:
f, ax = plt.subplots(1, 1, figsize=(7, 7))
ax.hist(allstar['gaia_G_corr'] - allstar['magroi'], bins=50)
ax.set_xlabel('Gaia Magnitude with Vignetting Correction - ROI star magnitude')
ax.set_title(f'DayObs {day_obs} SeqNum {seq_num_lo}-{seq_num_hi}')
f.savefig(os.path.join(output_dir, f'guider_mag_difference_hist_{day_obs}_{seq_num_lo}-{seq_num_hi}.png'))

### ROI Location Diagnostics

In [ ]:
f, ax = plt.subplots(1, 1)
ax.scatter(table['ampxLL'] - table['ampxLLH'], table['ampyLL'] - table['ampyLLH'],
           s=20., c=table['ccdColor'])
ax.set_xlim(-100, 100)
ax.set_ylim(-100, 100)
ax.set_aspect('equal')
ax.set_title('Difference in ROI LL location gROI - Header')
ax.set_xlabel('delta X [pix]')
ax.set_ylabel('delta Y [pix]')
for det, color in det_colors.items():
    ax.scatter([], [], c=[color], s=50, label=det)
f.legend()
f.savefig(os.path.join(output_dir, f'guider_roi_ll_groi_vs_header_{day_obs}_{seq_num_lo}-{seq_num_hi}.png'))

In [ ]:
f, ax = plt.subplots(1, 1)
ax.scatter(table['xroi'], table['yroi'], s=20., c=table['ccdColor'])
ax.set_xlim(0, 400)
ax.set_ylim(0, 400)
ax.set_aspect('equal')
ax.set_title('Y vs. X, Stars Found, ROI')
ax.set_xlabel('X [pix]')
ax.set_ylabel('Y [pix]')
for det, color in det_colors.items():
    ax.scatter([], [], c=[color], s=50, label=det)
f.legend()
f.savefig(os.path.join(output_dir, f'guider_roi_xy_foundstars_{day_obs}_{seq_num_lo}-{seq_num_hi}.png'))

### Star Position: Measured vs. Catalog

In [ ]:
f, ax = plt.subplots(1, 2, figsize=(11, 5))
ax[0].scatter(tabf['xccd'], tabf['ccdx'], s=20., c=tabf['ccdColor'])
ax[0].set_xlabel('Measured Star X CCD')
ax[0].set_ylabel('Guider Catalog X CCD')
ax[1].scatter(tabf['yccd'], tabf['ccdy'], s=20., c=tabf['ccdColor'])
ax[1].set_xlabel('Measured Star Y CCD')
ax[1].set_ylabel('Guider Catalog Y CCD')
f.suptitle('Found Stars: Measured vs. Catalog CCD Position')
f.savefig(os.path.join(output_dir, f'guider_ccd_measured_vs_catalog_found_{day_obs}_{seq_num_lo}-{seq_num_hi}.png'))

In [ ]:
f, ax = plt.subplots(1, 2, figsize=(11, 5))
ax[0].scatter(tabvol['ccdx'], tabvol['ccdy'], s=20., c=tabvol['ccdColor'])
ax[0].set_xlabel('Catalog X CCD')
ax[0].set_ylabel('Catalog Y CCD')
ax[0].set_aspect('equal')
ax[1].scatter(tabvol['xccd'], tabvol['yccd'], s=20., c=tabvol['ccdColor'])
ax[1].set_xlabel('Measured X CCD')
ax[1].set_ylabel('Measured Y CCD')
ax[1].set_aspect('equal')
f.suptitle('Volunteer Stars')
f.savefig(os.path.join(output_dir, f'guider_ccd_volunteer_positions_{day_obs}_{seq_num_lo}-{seq_num_hi}.png'))

### Position Residuals (Pointing-Corrected)

In [ ]:
f, ax = plt.subplots(1, 2, figsize=(12, 5))

ax[0].scatter(tabf['xccd'] - (tabf['ccdx'] + tabf['delta_xccd']),
              tabf['yccd'] - (tabf['ccdy'] + tabf['delta_yccd']),
              s=10., c=tabf['ccdColor'])
ax[0].set_xlim(-100, 100)
ax[0].set_ylim(-100, 100)
ax[0].set_aspect('equal')
ax[0].set_title('Catalog Star Found')
ax[0].set_xlabel('X residual [pix]')
ax[0].set_ylabel('Y residual [pix]')
for det, color in det_colors.items():
    ax[0].scatter([], [], c=[color], s=50, label=det)
ax[0].legend(fontsize=6)

ax[1].scatter(tabvol['xccd'] - (tabvol['ccdx'] + tabvol['delta_xccd']),
              tabvol['yccd'] - (tabvol['ccdy'] + tabvol['delta_yccd']),
              s=20., c=tabvol['ccdColor'])
ax[1].set_xlim(-300, 300)
ax[1].set_ylim(-300, 300)
ax[1].set_aspect('equal')
ax[1].set_title('ROI Volunteer')
ax[1].set_xlabel('X residual [pix]')
ax[1].set_ylabel('Y residual [pix]')

f.suptitle('Star in ROI - Catalog, Pointing Corrected, CCD Coords')
f.savefig(os.path.join(output_dir, f'guider_residual_found_vs_volunteer_{day_obs}_{seq_num_lo}-{seq_num_hi}.png'))

### Pointing Offsets

In [ ]:
f, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].scatter(table['delta_x'], table['delta_y'], s=20.)
ax[0].set_aspect('equal')
ax[0].set_xlabel('Delta X, Pointing offset [pix]')
ax[0].set_ylabel('Delta Y, Pointing offset [pix]')
ax[1].hist(table['delta_rot'], bins=50)
ax[1].set_xlabel('Delta Rotation, Pointing Offset [arcsec]')
f.savefig(os.path.join(output_dir, f'guider_pointing_offsets_{day_obs}_{seq_num_lo}-{seq_num_hi}.png'))

In [ ]:
f, ax = plt.subplots(1, 2, figsize=(12, 5))

ax[0].scatter(tabf['delta_rot'],
              tabf['xccd'] - (tabf['ccdx'] + tabf['delta_xccd']),
              s=10., c=tabf['ccdColor'])
ax[0].set_ylim(-100, 100)
ax[0].set_title('X residual vs Delta Rot')
ax[0].set_ylabel('X residual [pix]')
ax[0].set_xlabel('Delta Rot [arcsec]')
for det, color in det_colors.items():
    ax[0].scatter([], [], c=[color], s=50, label=det)
ax[0].legend(fontsize=6)

ax[1].scatter(tabf['delta_rot'],
              tabf['yccd'] - (tabf['ccdy'] + tabf['delta_yccd']),
              s=10., c=tabf['ccdColor'])
ax[1].set_ylim(-100, 100)
ax[1].set_title('Y residual vs Delta Rot')
ax[1].set_ylabel('Y residual [pix]')
ax[1].set_xlabel('Delta Rot [arcsec]')

f.suptitle('Position Residual vs. Rotation Offset, Found Stars')
f.savefig(os.path.join(output_dir, f'guider_residual_vs_deltarot_{day_obs}_{seq_num_lo}-{seq_num_hi}.png'))

### Residuals with Delta Rotation Cut

In [ ]:
tabfc = tabf[(tabf['delta_rot'] > -100)]

f, ax = plt.subplots(1, 2, figsize=(12, 5))

ax[0].scatter(tabfc['xccd'] - (tabfc['ccdx'] + tabfc['delta_xccd']),
              tabfc['yccd'] - (tabfc['ccdy'] + tabfc['delta_yccd']),
              s=10., c=tabfc['ccdColor'])
ax[0].set_xlim(-100, 100)
ax[0].set_ylim(-100, 100)
ax[0].set_aspect('equal')
ax[0].set_title('Catalog Star Found')
ax[0].set_xlabel('X residual [pix]')
ax[0].set_ylabel('Y residual [pix]')
for det, color in det_colors.items():
    ax[0].scatter([], [], c=[color], s=50, label=det)
ax[0].legend(fontsize=6)

ax[1].scatter(tabvol['xccd'] - (tabvol['ccdx'] + tabvol['delta_xccd']),
              tabvol['yccd'] - (tabvol['ccdy'] + tabvol['delta_yccd']),
              s=20., c=tabvol['ccdColor'])
ax[1].set_xlim(-300, 300)
ax[1].set_ylim(-300, 300)
ax[1].set_aspect('equal')
ax[1].set_title('ROI Volunteer')
ax[1].set_xlabel('X residual [pix]')
ax[1].set_ylabel('Y residual [pix]')

f.suptitle('Residuals, Pointing Corrected, Cut on delta_rot > -100 arcsec')
f.savefig(os.path.join(output_dir, f'guider_residual_deltarot_cut_{day_obs}_{seq_num_lo}-{seq_num_hi}.png'))

### Per-CCD Residual Mosaic

In [ ]:
layout = [
    [      ".", "R40_SG1", "R44_SG0",       "."],
    ["R40_SG0",       ".",       ".", "R44_SG1"],
    ["R00_SG1",       ".",       ".", "R04_SG0"],
    [      ".", "R00_SG0", "R04_SG1",       "."],
]

tabfc = tabf[(tabf['delta_rot'] > -100)]

fig, axs = plt.subplot_mosaic(layout, figsize=(7, 7))
for detector in camera:
    if detector.getType() == cameraGeom.DetectorType.GUIDER:
        detName = detector.getName()
        selccd = (tabfc['ccdName'] == detName)
        tab1 = tabfc[selccd]
        axs[detName].scatter(
            tab1['xccd'] - (tab1['ccdx'] + tab1['delta_xccd']),
            tab1['yccd'] - (tab1['ccdy'] + tab1['delta_yccd']),
            s=10.)
        axs[detName].set_xlim(-40, 40)
        axs[detName].set_ylim(-40, 40)
        axs[detName].set_aspect('equal')
        axs[detName].tick_params(axis='both', labelsize=8)

fig.suptitle('Star in ROI - Catalog, CCD coord, Pointing Corr., Cut on delta_rot')
fig.savefig(os.path.join(output_dir, f'guider_residual_perccd_mosaic_{day_obs}_{seq_num_lo}-{seq_num_hi}.png'))